# 03 — Industry Involvement

Materialises the dimension + bronze → silver → gold tables in
[`../specifications/03-industry-involvement.md`](../specifications/03-industry-involvement.md).

**Capability tables**
- `volume_forecast_dim_industrial_sites` (large C&I sites, owned here)
- `volume_forecast_bronze_site_telemetry`, `volume_forecast_bronze_production_schedule`
- `volume_forecast_silver_industrial_load` (baseline vs actual per site × interval)
- `volume_forecast_gold_industrial_flexibility` (dispatchable up/down MW — feeds the short-term DSR desk)

**UC comments:** [`uc_table_comments.py`](./uc_table_comments.py) — applied in the final cell.

In [ ]:
import os
import math
import random
import datetime as dt

from pyspark.sql import functions as F
from pyspark.sql import Row

dbutils.widgets.text("catalog", os.environ.get("DEMO_UC_CATALOG", "energy_utilities"))
dbutils.widgets.text("schema", os.environ.get("DEMO_UC_SCHEMA", "energy_trading2"))

CATALOG = dbutils.widgets.get("catalog").strip() or "energy_utilities"
SCHEMA = dbutils.widgets.get("schema").strip() or "energy_trading2"
print(f"Target: {CATALOG}.{SCHEMA}")
spark.sql(f"USE `{CATALOG}`.`{SCHEMA}`")


def fq(name: str) -> str:
    return f"`{CATALOG}`.`{SCHEMA}`.`{name}`"


random.seed(3)

TODAY = dt.date.today()
# Rolling horizon: 2 history days, today, 2 forecast days (matches 04 / 01 / 05 / 06).
DELIVERY_DATES = [TODAY + dt.timedelta(days=d) for d in (-2, -1, 0, 1, 2)]
LATEST_DATE = DELIVERY_DATES[-1]
N_INTERVALS = 96
NOW_INDEX = 56
NOW_TS = dt.datetime.combine(TODAY, dt.time(0, 0)) + dt.timedelta(minutes=15 * NOW_INDEX)


def interval_ts(day: dt.date, idx: int) -> dt.datetime:
    return dt.datetime.combine(day, dt.time(0, 0)) + dt.timedelta(minutes=15 * idx)


def is_settled(day: dt.date, idx: int) -> bool:
    if day < TODAY:
        return True
    if day == TODAY:
        return idx < NOW_INDEX
    return False


# ---- Dimension: large industrial / C&I sites (owned here) ----
# (site_id, name, industry, zone, contracted_mw, driver_type, is_flexible, flex_product)
SITES = [
    ("SMELT_DE_001",   "Rhine Aluminium Smelter", "ALUMINIUM",   "DE", 180.0, "CONTINUOUS",          True,  "SHED"),
    ("CHEM_DE_002",    "Ruhr Chemicals Park",     "CHEMICALS",   "DE", 120.0, "PRODUCTION_SCHEDULE", True,  "SHIFT"),
    ("DATACTR_NL_001", "Amsterdam Data Centre",   "DATA_CENTRE", "NL",  90.0, "CONTINUOUS",          False, None),
    ("STEEL_FR_001",   "Lorraine Steel Works",    "STEEL",       "FR", 150.0, "PRODUCTION_SCHEDULE", True,  "SHIFT"),
    ("COLD_BE_001",    "Antwerp Cold Storage",    "COLD_STORE",  "BE",  40.0, "WEATHER",             True,  "SHED"),
    ("CHEM_AT_001",    "Linz Chemicals",          "CHEMICALS",   "AT",  70.0, "PRODUCTION_SCHEDULE", False, None),
]
site_rows = [Row(site_id=s[0], site_name=s[1], industry=s[2], zone_code=s[3],
                 contracted_mw=s[4], driver_type=s[5], is_flexible=s[6]) for s in SITES]
spark.createDataFrame(site_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_dim_industrial_sites"))


def baseline_mw(site, idx: int) -> float:
    _, _, _, _, contracted, driver, _, _ = site
    h = idx * 15 / 60.0
    if driver == "CONTINUOUS":
        return contracted * (0.85 + 0.05 * math.sin(h / 24.0 * 2 * math.pi))
    if driver == "WEATHER":
        return contracted * (0.55 + 0.30 * max(0.0, math.sin(math.pi * (h - 6.0) / 16.0)))
    # PRODUCTION_SCHEDULE: day + swing shifts run hot, night low
    if 6 <= h < 22:
        return contracted * (0.80 if 6 <= h < 14 else 0.65)
    return contracted * 0.30

In [ ]:
# ---- Bronze: production schedule (per day / site / shift) ----
SHIFTS = [("DAY", range(24, 56)), ("SWING", range(56, 88)), ("NIGHT", list(range(88, 96)) + list(range(0, 24)))]
sched_rows = []
for day in DELIVERY_DATES:
    for site in SITES:
        for shift_name, shift_idx in SHIFTS:
            expected = sum(baseline_mw(site, i) for i in shift_idx) / max(1, len(list(shift_idx)))
            sched_rows.append(Row(
                ingestion_ts=dt.datetime.combine(day, dt.time(5, 0)),
                delivery_date=day, site_id=site[0], shift=shift_name,
                planned_output_units=round(expected * random.uniform(8.0, 12.0), 1),
                expected_mw=round(expected, 2),
            ))
spark.createDataFrame(sched_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_production_schedule"))

# ---- Bronze: site telemetry + Silver: baseline vs actual ----
telem_rows, load_rows = [], []
for day in DELIVERY_DATES:
    for site in SITES:
        for idx in range(N_INTERVALS):
            base = baseline_mw(site, idx)
            settled = is_settled(day, idx)
            actual = base * (1 + random.gauss(0, 0.04)) if settled else None
            start = interval_ts(day, idx)
            if actual is not None:
                status = "MAINTENANCE" if random.random() < 0.01 else ("REDUCED" if actual < 0.5 * site[4] else "RUNNING")
                if status == "MAINTENANCE":
                    actual = 0.0
                telem_rows.append(Row(
                    ingestion_ts=start, telemetry_ts=start, site_id=site[0],
                    actual_mw=round(actual, 2), status=status,
                ))
            load_rows.append(Row(
                delivery_date=day, interval_start=start, site_id=site[0], zone_code=site[3],
                baseline_mw=round(base, 2),
                actual_mw=round(actual, 2) if actual is not None else None,
                deviation_mw=round(actual - base, 2) if actual is not None else None,
            ))
spark.createDataFrame(telem_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_bronze_site_telemetry"))
spark.createDataFrame(load_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_silver_industrial_load"))

print("schedule:", spark.table(fq("volume_forecast_bronze_production_schedule")).count())
print("telemetry:", spark.table(fq("volume_forecast_bronze_site_telemetry")).count())
print("industrial_load:", spark.table(fq("volume_forecast_silver_industrial_load")).count())

In [ ]:
# ---- Gold: dispatchable industrial flexibility (flexible sites only) ----
flex_rows = []
for day in DELIVERY_DATES:
    for site in SITES:
        site_id, _, industry, zone, contracted, driver, is_flex, flex_product = site
        if not is_flex:
            continue
        for idx in range(N_INTERVALS):
            base = baseline_mw(site, idx)
            start = interval_ts(day, idx)
            down = base * (0.30 if flex_product == "SHED" else 0.20)
            up = base * (0.10 if flex_product == "SHIFT" else 0.0)
            is_peak = 8 <= idx * 15 // 60 < 20
            price = (140.0 if is_peak else 70.0) + random.uniform(-10, 10)
            if day < LATEST_DATE or start < NOW_TS:
                status = "AVAILABLE"
            elif (start - NOW_TS) <= dt.timedelta(hours=1):
                status = "ARMED"
            else:
                status = "AVAILABLE"
            if random.random() < 0.03:
                status = "UNAVAILABLE"
                up, down = 0.0, 0.0
            flex_rows.append(Row(
                delivery_date=day, interval_start=start, site_id=site_id, zone_code=zone,
                flex_product=flex_product,
                flexible_up_mw=round(up, 2), flexible_down_mw=round(down, 2),
                activation_price_eur_mwh=round(price, 2),
                min_duration_min=60 if flex_product == "SHED" else 120,
                availability_status=status,
            ))
spark.createDataFrame(flex_rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(fq("volume_forecast_gold_industrial_flexibility"))

display(spark.table(fq("volume_forecast_gold_industrial_flexibility"))
        .filter(F.col("delivery_date") == F.lit(LATEST_DATE))
        .groupBy("site_id", "flex_product")
        .agg(F.round(F.avg("flexible_down_mw"), 1).alias("avg_down_mw"),
             F.round(F.avg("flexible_up_mw"), 1).alias("avg_up_mw"))
        .orderBy("site_id"))

In [ ]:
# Row counts + Unity Catalog comments.
for t in [
    "volume_forecast_dim_industrial_sites",
    "volume_forecast_bronze_site_telemetry",
    "volume_forecast_bronze_production_schedule",
    "volume_forecast_silver_industrial_load",
    "volume_forecast_gold_industrial_flexibility",
]:
    print(f"  {t:48s}  {spark.table(fq(t)).count():>10,} rows")

from pathlib import Path

_uc_paths = []
try:
    _nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _uc_paths.append(Path(_nb).parent / "uc_table_comments.py")
except Exception:
    pass
_uc_paths.append(Path.cwd() / "uc_table_comments.py")

_uc_py = next((p for p in _uc_paths if p.is_file()), None)
if _uc_py is None:
    raise FileNotFoundError("uc_table_comments.py not found next to this notebook.")

exec(_uc_py.read_text(), globals())
apply_volume_forecast_notebook_03_comments(spark, CATALOG, SCHEMA)
print("UC comments applied for notebook 03.")